# Chapter 5 – Prompt Engineering Fundamentals
## Hands-On Colab Notebook for Network Engineers

**The four-hour lesson this notebook gives you in thirty minutes:**

> *"I wrote a prompt: 'Review this firewall change request.'*
> *The model said: 'Looks reasonable. Maybe test in a lab.'*
> *Completely useless. Four hours of trial and error later,*
> *I had a prompt that worked. A LLM does exactly what you tell it — not what you mean."*

| # | Demo | What it teaches |
|---|------|-----------------|
| 1 | The Specificity Problem | Vague ACL prompt vs specific prompt — dramatic difference |
| 2 | Zero-Shot | Simple tasks, no examples needed |
| 3 | Few-Shot | Add examples → consistent format every time |
| 4 | Chain-of-Thought | "Think step by step" unlocks real reasoning |
| 5 | System Prompts | Define expertise once, affects every response |
| 6 | Prompt Templates | Reusable f-string functions for common tasks |
| 7 | Structured JSON Output | Extract machine-readable data reliably |
| 8 | Prompt Chaining | Multi-step pipelines: output of one → input of next |
| 9 | Prompt Testing | Test prompts like you test network designs |

**~30 minutes** | **~$0.08 in API calls** | **No local setup required**

> Networking analogy: a prompt is a route-map.
> Vague prompt = `permit any, set local-preference random`.
> Specific prompt = explicit match conditions and set actions.


---
## Setup

In [ ]:
!pip install -q anthropic
print("Anthropic SDK installed")


In [ ]:
import os, json
from getpass import getpass
from anthropic import Anthropic

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("API key loaded from Colab Secrets")
except Exception:
    os.environ["ANTHROPIC_API_KEY"] = getpass("Paste your Anthropic API key: ")

client = Anthropic()
HAIKU  = "claude-haiku-4-5-20251001"    # fast, cheap – classification tasks
SONNET = "claude-sonnet-4-20250514"     # workhorse  – analysis, reasoning

def ask(prompt, model=HAIKU, max_tokens=300, system=None, temperature=0):
    """Minimal wrapper – one function used throughout this notebook."""
    kwargs = dict(model=model, max_tokens=max_tokens, temperature=temperature,
                  messages=[{"role": "user", "content": prompt}])
    if system:
        kwargs["system"] = system
    r = client.messages.create(**kwargs)
    return r.content[0].text

print(f"Ready. Haiku: {HAIKU.split('-')[1]} | Sonnet: {SONNET.split('-')[1]}")

---
## Demo 1 – The Specificity Problem

The most common prompt engineering mistake: asking a vague question and hoping the model guesses your intent.

**The rule:** A LLM does not know your network. It does not know your platform,
your IP scheme, your naming conventions, your compliance requirements.
**Everything it doesn't know — it guesses.**
Your prompt is how you replace guessing with knowledge.

**Route-map analogy:**
A vague prompt is like `permit any, set local-preference random`.
You will get output — just not the output you wanted.
A specific prompt is like an explicit match + set. Predictable. Testable.


In [ ]:
# The classic mistake: vague ACL request
VAGUE_PROMPT = "Generate an ACL to block traffic from 192.168.1.0/24 to port 80"

print("VAGUE PROMPT")
print("=" * 62)
print(f"Prompt: {VAGUE_PROMPT}")
print()
print("Output:")
print(ask(VAGUE_PROMPT, max_tokens=200))
print()
print("Problems with this output:")
print("  ❌ 'ip' used instead of 'tcp' — port 80 is TCP, not IP")
print("  ❌ No platform context — IOS? NX-OS? ASA? Palo Alto?")
print("  ❌ Wrong direction — may block source port 80, not destination")
print("  ❌ No verification steps, no interface application")


In [ ]:
# The specific version – fill in everything the model cannot know
SPECIFIC_PROMPT = """Generate a Cisco IOS-XE numbered extended ACL.

Intent: Deny HTTP (port 80) traffic from subnet 192.168.1.0/24 to any destination.
ACL number: 150
Platform: Cisco IOS-XE
Direction: Apply inbound on the interface where 192.168.1.0/24 traffic enters.

Requirements:
- Use 'tcp' protocol (not 'ip') since port 80 is TCP
- Add a remark comment explaining each rule
- End with explicit permit ip any any to avoid implicit deny cutting other traffic
- Provide the interface application command
- Provide verification commands after applying

Return config commands only, in a code block."""

print("SPECIFIC PROMPT")
print("=" * 62)
print("Output:")
print(ask(SPECIFIC_PROMPT, model=SONNET, max_tokens=400))
print()
print("Lesson: the model did not get smarter. YOU got more specific.")
print("Every word you omit is a choice the model makes for you.")


In [ ]:
# Iterative refinement — the three versions of any prompt

prompts = {
    "v1 (too vague)":    "How do I fix my network?",
    "v2 (specific)":     "OSPF neighbor stuck in EXSTART state. Interface MTU is 9000 on R1 and 1500 on R2. How do I fix it?",
    "v3 (persona+format)":"You are a Cisco IOS-XE expert. OSPF neighbor stuck in EXSTART. MTU mismatch: R1=9000, R2=1500. Provide: root cause (one sentence), fix command for each router, verification command.",
}

print("ITERATIVE REFINEMENT – same problem, three prompt versions")
print("=" * 62)
for label, prompt in prompts.items():
    print(f"\n[{label}]")
    print(f"Prompt: {prompt[:80]}")
    print(f"Output: {ask(prompt, max_tokens=120).strip()[:200]}")
    print()

print("Notice: v3 gives a structured, actionable answer in the fewest tokens.")
print("Better prompts = better output AND lower cost (shorter responses).")


---
## Demo 2 – Zero-Shot: No Examples Needed

**Zero-shot**: give the model an instruction and input — no examples.
Works for well-understood concepts where the model's training is sufficient.

**Best for**: protocol explanations, quick config explanations, FAQ, first-pass log triage.
**Fails for**: custom formats, org-specific terminology, novel classification schemes → use few-shot.

**Cost tip**: Haiku + zero-shot is the cheapest possible AI call.
Use this for everything that does not need examples or reasoning.


In [ ]:
# Zero-shot: protocol explanation — simple, common, model knows this well
print("ZERO-SHOT: Protocol FAQ")
print("=" * 55)

faq_questions = [
    "What is the OSPF dead interval and why does it matter?",
    "BGP weight attribute: higher or lower is preferred?",
    "What does 'ip ospf network point-to-point' do?",
]

for q in faq_questions:
    answer = ask(q, max_tokens=80)
    print(f"Q: {q}")
    print(f"A: {answer.strip()}")
    print()


In [ ]:
# Zero-shot: config explanation — works great for common IOS config blocks
config_snippet = """
interface GigabitEthernet0/1
 ip address 10.1.1.1 255.255.255.252
 ip ospf network point-to-point
 ip ospf cost 10
 bfd interval 100 min_rx 100 multiplier 3
 no shutdown
"""

prompt = f"Explain what this Cisco IOS config does. Be brief — 3 bullet points max.\n{config_snippet}"

print("ZERO-SHOT: Config Explanation")
print("=" * 55)
print("Config:")
print(config_snippet)
print("Explanation:")
print(ask(prompt, max_tokens=200))


---
## Demo 3 – Few-Shot: Show Examples, Get Consistent Output

**Few-shot**: provide 2–5 input/output examples before the actual question.
The model learns the pattern and applies it — including YOUR format, YOUR terminology.

**When zero-shot format is inconsistent**: a few examples anchor the output.
The model's zero-shot might return "The VLAN is 175" or "VLAN: one seven five".
A few-shot example forces it to return just "175".

**Networking analogy:** onboarding a new engineer.
You don't write a 50-page procedures manual.
You show them 3 examples of how you handle tickets. They get the pattern.
Same with LLMs.


In [ ]:
# Side-by-side: zero-shot vs few-shot on log parsing
# Task: extract VLAN ID from different log message formats

LOG_TO_PARSE = '"%VLAN_MGR-3-VLAN_STATE: VLAN 175 enabled"'

# Zero-shot — model might add extra words
ZERO_SHOT = f"""Extract the VLAN ID from this network log entry. Return only the number.

Log: {LOG_TO_PARSE}
VLAN:"""

# Few-shot — examples enforce exact format
FEW_SHOT = f"""Extract VLAN IDs from network log entries. Return only the number.

Example 1:
Log: "%LINK-3-UPDOWN: Interface Vlan100, changed state to up"
VLAN: 100

Example 2:
Log: "%SYS-5-CONFIG: Configured from console by admin on vlan 250"
VLAN: 250

Example 3:
Log: "STP: VLAN0050 Port Gi1/0/1 is now in forwarding state"
VLAN: 50

Now extract from this:
Log: {LOG_TO_PARSE}
VLAN:"""

print("ZERO-SHOT vs FEW-SHOT – VLAN Extraction")
print("=" * 58)
print(f"Log: {LOG_TO_PARSE}")
print()

zs = ask(ZERO_SHOT, max_tokens=15)
fs = ask(FEW_SHOT,  max_tokens=10)
print(f"Zero-shot output: '{zs.strip()}'")
print(f"Few-shot output:  '{fs.strip()}'")
print()
print("Few-shot guarantees the model returns just the number.")
print("Zero-shot might add 'The VLAN is' or 'VLAN ID: 175' — breaks parsers.")


In [ ]:
# Real-world few-shot: teach the model YOUR organization's interface naming convention
# The model doesn't know your org's standards — you teach it with 4 examples

NAMING_PROMPT = """Convert interface names to our organization's short format.

Examples:
Full name: GigabitEthernet0/0/1    Short: Gi0/0/1
Full name: TenGigabitEthernet1/2   Short: Te1/2
Full name: Bundle-Ether100         Short: BE100
Full name: Port-channel50          Short: Po50
Full name: FastEthernet0/24        Short: Fa0/24

Now convert:
Full name: HundredGigabitEthernet0/0/0/1
Short:"""

print("FEW-SHOT: Teaching Org-Specific Naming Convention")
print("=" * 58)
result = ask(NAMING_PROMPT, max_tokens=15)
print(f"Output: '{result.strip()}'")
print()
print("The model learned Hu0/0/0/1 from the pattern in the examples.")
print("This works for ticket formats, severity labels, anything org-specific.")
print()
print("Rule of thumb:")
print("  2-3 examples → teaches the basic pattern")
print("  4-5 examples → handles edge cases and unusual inputs")


---
## Demo 4 – Chain-of-Thought: Force the Model to Reason

**Chain-of-Thought (CoT)**: ask the model to show its reasoning step by step.
Instead of jumping to conclusions, it walks through the problem systematically.

**When to use it**: Complex troubleshooting, multi-variable analysis,
root cause investigation — anything where the reasoning process matters.

**The magic phrase**: "Think step by step" / "Diagnose step by step" /
"Analyze layer by layer" — these phrases alone trigger reasoning mode.

**Networking analogy:** the difference between a junior engineer who shouts
"it's the MTU!" and a senior engineer who checks physical → data link →
network → transport before concluding. The reasoning IS the value.

**Cost note**: CoT outputs are 3–10× longer. Use Sonnet (not Opus) for most troubleshooting.


In [ ]:
# Direct answer vs Chain-of-Thought on a BGP problem with a non-obvious root cause

BGP_PROBLEM = """
Router 1 config:
  router bgp 65001
   neighbor 10.1.1.2 remote-as 65002
   neighbor 10.1.1.2 update-source Loopback0

Router 1 Loopback0:
  interface Loopback0
   ip address 10.1.1.1 255.255.255.255

Router 2 config:
  router bgp 65002
   neighbor 10.1.1.1 remote-as 65001

Symptom: BGP session stuck in Active state. AS numbers, IPs, and routing are correct.
"""

# Direct answer – model pattern-matches to the most common BGP issue
DIRECT  = f"Why is this BGP session stuck in Active? One sentence.\n{BGP_PROBLEM}"

# Chain-of-Thought – model walks through each possible cause
COT = f"""Diagnose why this BGP session is stuck in Active. Think step by step.

{BGP_PROBLEM}

Check each possible cause in order:
1. AS numbers – does R1 expect the right AS from R2?
2. Neighbor IPs – are R1 and R2 configured to reach each other?
3. Update-source – does R1 send BGP OPEN from Loopback0? Does R2 expect that?
4. Root cause – one specific sentence."""

print("CHAIN-OF-THOUGHT vs DIRECT ANSWER – BGP troubleshooting")
print("=" * 62)
print("DIRECT ANSWER:")
print(ask(DIRECT, model=HAIKU, max_tokens=80).strip())
print()
print("CHAIN-OF-THOUGHT (Sonnet):")
print(ask(COT, model=SONNET, max_tokens=400).strip())
print()
print("The CoT version finds the real root cause: asymmetric update-source.")
print("R1 sends BGP OPEN from 10.1.1.1 (Loopback), R2 doesn't configure update-source,")
print("so R2 sends from its physical interface — TCP session can never establish.")


In [ ]:
# OSI-layer systematic troubleshooting template
# Structure forces the model through every layer before concluding

OSI_TEMPLATE = """Troubleshoot this network issue layer by layer. Think step by step.

Problem: {problem}

Analyze each OSI layer:
1. Layer 1 (Physical): What interface/cable issues could cause this?
2. Layer 2 (Data Link): What VLAN, STP, or MAC issues could cause this?
3. Layer 3 (Network):  What routing, IP, or ACL issues could cause this?
4. Layer 4+ (Transport/App): What port blocking or NAT issues apply?

For each layer: state whether it is a likely cause and why.
Final: most probable root cause (one sentence) and first command to run."""

PROBLEM = "User on VLAN 20 can ping the gateway but cannot reach the internet."

print("OSI LAYER-BY-LAYER CoT TEMPLATE")
print("=" * 62)
print(ask(OSI_TEMPLATE.format(problem=PROBLEM), model=SONNET, max_tokens=500).strip())


---
## Demo 5 – System Prompts: Define the Expert Once

**System prompt**: a persistent instruction that sets the model's identity and behavior
for the entire session. Applied once, affects every response.

**When to use it**: interactive tools, NOC chatbots, any multi-turn interaction where
you want consistent expertise, format, and terminology.

**BGP neighbor analogy**: the system prompt is your `neighbor` configuration.
You configure the session parameters once — authentication, route-maps, timers.
Every prefix exchange that follows happens within those parameters.

**Practical tip**: put naming conventions, output format, and terminology preferences
in the system prompt — not in every user message. Define once, reuse everywhere.


In [ ]:
# The dramatic difference a system prompt makes on config review

SAMPLE_CONFIG = """
hostname edge-rtr-01
enable password cisco123
snmp-server community public RO
snmp-server community private RW
line vty 0 4
 transport input telnet
 no login
"""

# Without system prompt – generic advice
NO_SYSTEM = f"Review this router config.\n{SAMPLE_CONFIG}"

# With system prompt – expert persona + strict output format
SYSTEM = """You are a senior network security engineer with 15 years of enterprise experience.
When analyzing configurations:
- Prioritize issues: CRITICAL → HIGH → MEDIUM → LOW
- Reference the exact config line that causes each issue
- Provide specific remediation commands
- Never suggest changes without a maintenance window warning
Format: Issue | Severity | Exact Line | Fix Command"""

print("WITHOUT system prompt:")
print("=" * 62)
print(ask(NO_SYSTEM, max_tokens=200).strip())
print()
print("WITH system prompt:")
print("=" * 62)
print(ask(NO_SYSTEM, system=SYSTEM, max_tokens=300).strip())
print()
print("Same model. Same config. Same question.")
print("The system prompt replaced 'generic advice' with 'structured security report'.")


In [ ]:
# Practical system prompt with terminology and output standards
# Define this once and reuse across your entire team's tools

TEAM_SYSTEM = """You are a network automation specialist for our enterprise team.

TERMINOLOGY STANDARDS (always use these):
- 'GigabitEthernet' not 'GigE' or 'Gi'
- 'trunk port' not 'tagged port'
- 'VLAN' (all caps) not 'vlan' or 'Virtual LAN'
- 'access-list' not 'ACL' in config commands

OUTPUT FORMAT:
- Config commands must be in code blocks (``` ``` ```)
- Issues listed as: [SEVERITY] Line: 'exact line' → Fix: 'exact command'
- Always end with: Verification: [show command to confirm fix]"""

test_q = "Is 'transport input telnet' on VTY lines a security risk?"

print("TEAM SYSTEM PROMPT – enforces consistent terminology and format")
print("=" * 62)
result = ask(test_q, system=TEAM_SYSTEM, max_tokens=200)
print(result.strip())
print()
print("Every team member using this system prompt gets the same output format.")
print("Build it once, paste it into every tool you write.")


---
## Demo 6 – Prompt Templates: Reusable f-string Functions

If you write the same prompt 3 times across different projects, it should be a function.

**Template = f-string function that returns a prompt string.**
The function builds the prompt. The `ask()` call sends it.
Keep them separate — easier to test, easier to share.

This is how you build a team prompt library that everyone uses consistently.


In [ ]:
# Three reusable prompt template functions for common networking tasks
# Each returns a string (the prompt) – NOT an API response

def log_classify_prompt(log_entry: str) -> str:
    """Zero-shot log classification. Use with Haiku for bulk processing."""
    return f"""Classify this network log entry.
Return ONLY one word: INFO, WARNING, ERROR, or CRITICAL.

Definitions:
- INFO: Normal state changes, audit events
- WARNING: Non-critical, suboptimal conditions
- ERROR: Service-affecting failures
- CRITICAL: Outages, security incidents, resource exhaustion

Log entry:
{log_entry}

Classification:"""


def acl_generate_prompt(intent: str, platform: str = "IOS-XE",
                        acl_name: str = None) -> str:
    """Generates a complete ACL with comments and verification steps."""
    name_part = f"Named ACL '{acl_name}'" if acl_name else "Numbered extended ACL"
    return f"""Generate a Cisco {platform} {name_part}.

Intent: {intent}
Platform: Cisco {platform}

Requirements:
- Use correct protocol (tcp/udp/icmp, NOT generic 'ip' where protocol matters)
- Add remark lines explaining each rule
- End with explicit deny with log
- Provide interface application and verification commands"""


def bgp_troubleshoot_prompt(r1_cfg: str, r2_cfg: str, symptom: str) -> str:
    """BGP troubleshooting with step-by-step chain-of-thought."""
    return f"""You are a BGP expert. The session is not establishing.

Symptom: {symptom}

Router 1:
{r1_cfg}

Router 2:
{r2_cfg}

Diagnose step by step:
1. AS numbers – match on both sides?
2. Neighbor IPs – correct and reachable?
3. Update-source – consistent?
4. Authentication – password/MD5 match?

Provide: root cause (one sentence) + fix commands for each router + verify command."""


# Show the prompt BEFORE sending – always a good habit
sample_log = "%OSPF-5-ADJCHG: Nbr 10.1.1.2 from FULL to DOWN, Dead timer expired"
print("PROMPT PREVIEW (log_classify_prompt):")
print("-" * 55)
print(log_classify_prompt(sample_log))


In [ ]:
# Use the template functions in real calls

print("TEMPLATE FUNCTIONS IN ACTION")
print("=" * 55)

# Log classification – bulk use case (Haiku, temperature 0)
test_logs = [
    "%OSPF-5-ADJCHG: Nbr 10.1.1.2 from FULL to DOWN, Dead timer expired",
    "%LINEPROTO-5-UPDOWN: Interface Gi0/1 changed to up",
    "%SEC_LOGIN-4-LOGIN_FAILED: Login failed from 192.168.1.100",
    "%SYS-5-CONFIG_I: Configured from console by admin",
]

print("Log Classification (Haiku, template function):")
for log in test_logs:
    prompt      = log_classify_prompt(log)
    result      = ask(prompt, model=HAIKU, max_tokens=10)
    print(f"  [{result.strip():<8}] {log[:70]}")

print()

# ACL generation – use a template with Sonnet
acl_prompt = acl_generate_prompt(
    intent="Block all Telnet (port 23) inbound from the internet. Allow SSH (port 22).",
    platform="IOS-XE",
    acl_name="MGMT-ACCESS"
)
print("ACL Generation (Sonnet, template function):")
print(ask(acl_prompt, model=SONNET, max_tokens=350))


---
## Demo 7 – Structured JSON Output: Machine-Readable Data

When you connect AI to other systems — ticket systems, CMDB, monitoring tools —
you need structured data, not free-form text.

**Two problems to solve:**
1. The model might wrap JSON in markdown code blocks (` ```json ... ``` `)
2. The model might generate invalid JSON on edge cases

**Solution:**
1. Strip markdown fences before parsing
2. Wrap `json.loads()` in `try/except` — never crash a production pipeline on AI output
3. Return `None` on failure — let the caller decide what to do

**Networking analogy:** always validate config syntax before pushing to device.
Never assume the AI output is valid without checking.


In [ ]:
import re

def extract_json(text: str) -> dict | None:
    """
    Safely extract JSON from LLM response.
    Handles: markdown code blocks, leading/trailing text, invalid syntax.
    Returns: parsed dict, or None on failure.
    """
    # Strip markdown code fences (```json ... ``` or ``` ... ```)
    text = re.sub(r"```(?:json)?\s*", "", text).replace("```", "").strip()

    # Find the first { and last } to handle leading/trailing text
    start = text.find("{")
    end   = text.rfind("}") + 1
    if start >= 0 and end > start:
        text = text[start:end]

    try:
        return json.loads(text)
    except json.JSONDecodeError as e:
        print(f"  JSON parse failed: {e}")
        print(f"  Raw text: {text[:200]}")
        return None   # NEVER crash the pipeline on bad AI output


# Demonstrate on a real security audit prompt that returns JSON
AUDIT_CONFIG = """
hostname core-rtr-01
snmp-server community public RO
line vty 0 4
 transport input telnet
 no login
"""

AUDIT_PROMPT = f"""Security audit this Cisco IOS config.
Return ONLY valid JSON — no markdown, no extra text:
{{
  "critical": [{{"issue": "...", "line": "...", "fix": "..."}}],
  "high": [],
  "medium": [],
  "low": [],
  "summary": "one-line overall assessment"
}}

Config:
{AUDIT_CONFIG}"""

print("STRUCTURED JSON OUTPUT – Security Audit")
print("=" * 58)
raw_output = ask(AUDIT_PROMPT, model=SONNET, max_tokens=500)
print("Raw AI output (first 200 chars):")
print(raw_output[:200])
print()

parsed = extract_json(raw_output)
if parsed:
    print("Parsed successfully!")
    print(f"  Critical issues: {len(parsed.get('critical', []))}")
    print(f"  Summary: {parsed.get('summary', 'N/A')}")
    for issue in parsed.get("critical", []):
        print(f"  [CRITICAL] {issue.get('issue')} → {issue.get('fix')}")
else:
    print("Parse failed – check raw output above")


In [ ]:
# JSON with retry: if parse fails, ask once more with stronger instruction

def ask_for_json(prompt: str, model=SONNET, max_tokens=500, retries=2) -> dict | None:
    """Ask for JSON, retry with stricter instruction if parse fails."""
    for attempt in range(retries):
        suffix = ""
        if attempt > 0:
            # Second attempt: add even stronger instruction
            suffix = "\n\nCRITICAL: Return ONLY the raw JSON object. No text before or after. No markdown."
        raw = ask(prompt + suffix, model=model, max_tokens=max_tokens)
        result = extract_json(raw)
        if result is not None:
            return result
        print(f"  Attempt {attempt+1} failed – retrying with stricter instruction")
    print("  All attempts failed – returning None")
    return None


# Test: log classification returning structured JSON
LOG_JSON_PROMPT = """Classify this syslog message and return ONLY JSON:
{
  "severity": "INFO|WARNING|ERROR|CRITICAL",
  "subsystem": "OSPF|BGP|STP|INTERFACE|SECURITY|OTHER",
  "needs_action": true|false,
  "summary": "one sentence"
}

Log: %OSPF-5-ADJCHG: Process 1, Nbr 10.1.1.2 from FULL to DOWN, Dead timer expired"""

print("JSON WITH RETRY – Log Classification")
print("=" * 55)
result = ask_for_json(LOG_JSON_PROMPT, model=HAIKU, max_tokens=150)
if result:
    for k, v in result.items():
        print(f"  {k}: {v}")


---
## Demo 8 – Prompt Chaining: Output of One Step → Input of Next

Complex tasks rarely fit in one prompt. When you try, the model starts dropping
parts of the instruction — it runs out of "attention" for all requirements.

**Solution**: chain prompts. Break the task into focused steps.
Output from step 1 becomes input to step 2.

**Benefits:**
- Each step is simple and focused — easier to debug
- Different models for different steps (Haiku extracts, Sonnet analyzes)
- Short-circuit early if a step produces unacceptable output
- Validate between steps — is the JSON valid? does it have required fields?

**Example: Change Request Review Pipeline**
```
Ticket text → [Step 1: Extract] → JSON changes → [Step 2: Security] → findings → [Step 3: Recommend]
                 Haiku                                  Sonnet
```


In [ ]:
# Two-step change review pipeline
# Step 1: Haiku extracts the structured changes (cheap, simple task)
# Step 2: Sonnet analyzes security issues (needs reasoning)

CHANGE_TICKET = """
Change Request #4821
Device: core-rtr-01
Requester: john.smith@company.com

Changes:
- Add SNMP community string 'monitoring123' for read-only access from 10.10.0.0/24
- Enable telnet on VTY lines 0-4 for temporary access during maintenance
- Add static route: ip route 0.0.0.0 0.0.0.0 203.0.113.1
"""

# ── Step 1: Extract changes (Haiku – cheap, structured extraction) ─────────────
step1_prompt = f"""Extract configuration changes from this ticket.
Return ONLY valid JSON:
{{
  "device": "...",
  "changes": [
    {{"type": "snmp|telnet|routing|acl|other", "description": "..."}}
  ]
}}

Ticket:
{CHANGE_TICKET}"""

print("PROMPT CHAINING – Change Request Review Pipeline")
print("=" * 62)
print("STEP 1: Extract changes (Haiku)")
raw1     = ask(step1_prompt, model=HAIKU, max_tokens=300)
step1_ok = extract_json(raw1)
if step1_ok:
    print(f"  Extracted {len(step1_ok.get('changes', []))} changes from ticket")
    for c in step1_ok.get("changes", []):
        print(f"  [{c['type'].upper():<10}] {c['description']}")
else:
    print("  Step 1 failed – stopping pipeline")
    step1_ok = None


In [ ]:
# ── Step 2: Security analysis (Sonnet – needs reasoning) ─────────────────────
if step1_ok:
    step2_prompt = f"""Security review these network change requests.

Changes to review:
{json.dumps(step1_ok.get('changes', []), indent=2)}

For each change, check:
- Is this a known security risk?
- Does it violate hardening best practices?
- Should it be APPROVED, APPROVED-WITH-CONDITIONS, or REJECTED?

Return ONLY valid JSON:
{{
  "findings": [
    {{"change_type": "...", "risk": "LOW|MEDIUM|HIGH|CRITICAL",
      "reason": "...", "decision": "APPROVED|APPROVED-WITH-CONDITIONS|REJECTED"}}
  ],
  "overall": "APPROVE|REJECT",
  "summary": "one sentence"
}}"""

    print("\nSTEP 2: Security analysis (Sonnet)")
    raw2     = ask(step2_prompt, model=SONNET, max_tokens=500)
    step2_ok = extract_json(raw2)

    if step2_ok:
        print(f"  Overall decision: {step2_ok.get('overall')}")
        print(f"  Summary: {step2_ok.get('summary')}")
        print()
        for f in step2_ok.get("findings", []):
            icon = "✅" if f["decision"] == "APPROVED" else "⚠️" if "CONDITIONS" in f["decision"] else "❌"
            print(f"  {icon} [{f['risk']:<8}] {f['change_type']}: {f['reason'][:60]}")
    else:
        print("  Step 2 parse failed")
else:
    print("Skipping Step 2 – Step 1 produced no output")

print()
print("Each step is simple and focused. Haiku does the cheap extraction.")
print("Sonnet does the expensive reasoning. Early exit if step 1 fails.")


---
## Demo 9 – Prompt Testing: Test Prompts Like Network Designs

A prompt that works on 5 examples might fail on the 6th.
The only way to know a prompt is reliable is to test it systematically.

**Networking analogy**: you don't design a BGP policy and hope it works.
You test with specific route advertisements, verify results match intent, then deploy.
Prompts deserve the same rigor.

**Simple test framework:**
- `test_cases`: list of (input, expected_string_in_output)
- Run each, check if `expected` appears in the response
- Print pass/fail report with percentage


In [ ]:
# Simple prompt test framework – no class needed, just a function

def run_prompt_tests(prompt_fn, test_cases, model=HAIKU, label=""):
    """
    Run a list of test cases against a prompt function.
    prompt_fn(input) → prompt string
    test_cases: list of (input, expected_substring, description)
    """
    passed = 0
    print(f"PROMPT TEST: {label}")
    print("=" * 62)
    print(f"Model: {model}  |  Tests: {len(test_cases)}")
    print()

    for inp, expected, desc in test_cases:
        prompt = prompt_fn(inp)
        actual = ask(prompt, model=model, max_tokens=20)
        ok     = expected.lower() in actual.lower()
        passed += ok
        icon   = "✅ PASS" if ok else "❌ FAIL"
        print(f"  {icon}  {desc}")
        if not ok:
            print(f"         Expected: '{expected}'  Got: '{actual.strip()[:50]}'")

    rate = passed / len(test_cases) * 100
    print()
    print(f"Result: {passed}/{len(test_cases)} passed ({rate:.0f}%)")
    return rate


# Test the log_classify_prompt from Demo 6
LOG_TEST_CASES = [
    ("%OSPF-5-ADJCHG: Nbr 10.1.1.2 from FULL to DOWN",   "CRITICAL", "OSPF neighbor down"),
    ("%LINK-3-UPDOWN: Interface Gi0/1, changed state to up",  "INFO",    "Interface coming up"),
    ("%SEC_LOGIN-4-LOGIN_FAILED: Login failed from 10.10.0.1", "ERROR",  "Failed login"),
    ("%SYS-5-CONFIG_I: Configured from console by admin",       "INFO",   "Config change event"),
    ("%BGP-3-NOTIFICATION: received from neighbor 10.1.1.2",    "ERROR",  "BGP notification"),
]

rate = run_prompt_tests(
    prompt_fn  = log_classify_prompt,
    test_cases = LOG_TEST_CASES,
    model      = HAIKU,
    label      = "Log Classification Prompt"
)


In [ ]:
# When tests fail: show the iterative improvement cycle

# v1 – test might fail because format is ambiguous
def classify_v1(log):
    return f"What is the severity of: {log}"

# v2 – constrain the output format
def classify_v2(log):
    return f"Classify severity. Return only: INFO, WARNING, ERROR, or CRITICAL.\n\n{log}"

# v3 – add definitions for edge cases
def classify_v3(log):
    return log_classify_prompt(log)   # our full template from Demo 6

print("PROMPT VERSION COMPARISON")
print("=" * 62)

for version, fn in [("v1 (vague)", classify_v1),
                    ("v2 (constrained)", classify_v2),
                    ("v3 (full template)", classify_v3)]:
    rate = run_prompt_tests(fn, LOG_TEST_CASES, model=HAIKU, label=version)
    print()

print("This is the improvement cycle:")
print("  Write → Test → See failures → Fix → Test again")
print("  Same workflow as iterative network design testing.")


---
## You Completed Chapter 5!

| Demo | Technique | Key rule |
|------|-----------|----------|
| 1 | Specificity | Every word omitted = a guess. Replace guesses with facts. |
| 2 | Zero-shot | Fast, cheap. Use for well-understood tasks with Haiku. |
| 3 | Few-shot | 3–5 examples anchor format and terminology. |
| 4 | Chain-of-Thought | "Think step by step" unlocks real reasoning on complex problems. |
| 5 | System prompts | Define expertise once. Every message benefits. |
| 6 | Prompt templates | f-string functions → reusable, testable, shareable. |
| 7 | Structured JSON | Strip markdown → `try/except json.loads()` → never crash on AI output. |
| 8 | Prompt chaining | One focused step at a time. Cheap model extracts, smart model reasons. |
| 9 | Prompt testing | Build test cases before deploying. Same rigor as network design testing. |

---

### The Decision Table

| Task | Technique | Model |
|------|-----------|-------|
| Protocol FAQ, quick explanation | Zero-shot | Haiku |
| Log classification, data extraction | Zero-shot + JSON | Haiku |
| Custom format, org-specific terminology | Few-shot | Haiku |
| Complex troubleshooting | Chain-of-Thought | Sonnet |
| Interactive tool, chatbot | System prompt | Sonnet |
| Multi-step workflow | Prompt chaining | Haiku → Sonnet |

### Networking Analogy Cheat Sheet

| Prompt concept | Networking equivalent |
|---|---|
| Prompt | BGP route-map |
| System prompt | `neighbor` configuration (session parameters) |
| Zero-shot | Static route — fast, no context needed |
| Few-shot | Onboarding a new engineer with 3 examples |
| Chain-of-Thought | OSI layer-by-layer troubleshooting |
| Prompt template | Config template (reuse with different variables) |
| Prompt testing | BGP policy validation in a test bed |
| JSON extraction | `show` command output parsing |

---

### What's Next?

**Chapter 6 – Multi-Turn Conversations and Context Management**
How to build conversational AI tools that maintain context across multiple exchanges,
manage context windows efficiently, and handle conversation state.

*Keep your prompt template functions from Demo 6 — they become the foundation for the tools in the next chapters.*
